# 04 - Modelagem com features derivadas do World Bank

Este notebook testa se a engenharia temporal aplicada aos indicadores socioeconômicos do World Bank melhora os modelos de previsão de conflito.

A comparação é feita no mesmo conjunto de países/anos do dataset integrado para manter a avaliação justa.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parents[1]

DATA_PATH = PROJECT_ROOT / "data" / "final" / "conflict_country_year_world_bank_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Countries:", df["country"].nunique())
print("Years:", df["year"].min(), "-", df["year"].max())
df.head()

Dataset loaded successfully.
Shape: (6663, 63)
Countries: 195
Years: 1989 - 2023


,country,year,region,main_government_name,state_based_conflict_exists,state_based_dyad_count,state_based_deaths_best,intrastate_conflict_exists,intrastate_deaths_best,interstate_conflict_exists,...,inflation_consumer_prices_annual_pct_change_1y,inflation_consumer_prices_annual_pct_rolling_3y_mean,unemployment_total_pct_missing,unemployment_total_pct_lag1,unemployment_total_pct_change_1y,unemployment_total_pct_rolling_3y_mean,military_expenditure_pct_gdp_missing,military_expenditure_pct_gdp_lag1,military_expenditure_pct_gdp_change_1y,military_expenditure_pct_gdp_rolling_3y_mean
0,Afghanistan,1989,Asia,Government of Afghanistan,1,5,5174,1,5174,0,...,NaN,NaN,1,NaN,NaN,NaN,1,NaN,NaN,NaN
1,Afghanistan,1990,Asia,Government of Afghanistan,1,5,1478,1,1478,0,...,NaN,NaN,1,NaN,NaN,NaN,1,NaN,NaN,NaN
2,Afghanistan,1991,Asia,Government of Afghanistan,1,4,3302,1,3302,0,...,NaN,NaN,0,NaN,NaN,7.972,1,NaN,NaN,NaN
3,Afghanistan,1992,Asia,Government of Afghanistan,1,4,4287,1,4287,0,...,NaN,NaN,0,7.972,-0.014,7.965,1,NaN,NaN,NaN
4,Afghanistan,1993,Asia,Government of Afghanistan,1,4,4071,1,4071,0,...,NaN,NaN,0,7.958,-0.050,7.946,1,NaN,NaN,NaN


In [2]:
TARGET_COLUMN = "target_conflict_next_year"
TRAIN_END_YEAR = 2016

BASE_FEATURE_COLUMNS = [
    "year",
    "state_based_conflict_exists",
    "state_based_dyad_count",
    "state_based_deaths_best",
    "intrastate_conflict_exists",
    "intrastate_deaths_best",
    "interstate_conflict_exists",
    "interstate_deaths_best",
    "non_state_conflict_exists",
    "non_state_dyad_count",
    "non_state_deaths_best",
    "one_sided_violence_exists",
    "one_sided_dyad_count",
    "one_sided_deaths_best",
    "cumulative_organized_violence_deaths_best",
    "organized_violence_exists",
]

TEMPORAL_FEATURE_COLUMNS = [
    "conflict_previous_year",
    "conflict_last_3_years_count",
    "conflict_last_5_years_count",
    "deaths_previous_year",
    "deaths_last_3_years_sum",
    "deaths_last_5_years_sum",
    "years_since_last_conflict",
]

WORLD_BANK_RAW_COLUMNS = [
    "population_total",
    "gdp_per_capita_current_usd",
    "gdp_growth_annual_pct",
    "inflation_consumer_prices_annual_pct",
    "unemployment_total_pct",
    "military_expenditure_pct_gdp",
]

WORLD_BANK_ENGINEERED_COLUMNS = [
    col
    for col in df.columns
    if (
        col.endswith("_missing")
        or col.endswith("_lag1")
        or col.endswith("_change_1y")
        or col.endswith("_rolling_3y_mean")
    )
]

EXPERIMENTS = {
    "temporal_only": BASE_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS,
    "temporal_world_bank_raw": BASE_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS + WORLD_BANK_RAW_COLUMNS,
    "temporal_world_bank_engineered": (
        BASE_FEATURE_COLUMNS
        + TEMPORAL_FEATURE_COLUMNS
        + WORLD_BANK_RAW_COLUMNS
        + WORLD_BANK_ENGINEERED_COLUMNS
    ),
}

print("Target distribution:")
print(df[TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

print("\nRaw World Bank features:", len(WORLD_BANK_RAW_COLUMNS))
print("Engineered World Bank features:", len(WORLD_BANK_ENGINEERED_COLUMNS))

print("\nMissing values in engineered World Bank features:")
print(
    df[WORLD_BANK_ENGINEERED_COLUMNS]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .head(20)
)

Target distribution:
target_conflict_next_year
0    0.7018
1    0.2982
Name: proportion, dtype: float64

Raw World Bank features: 6
Engineered World Bank features: 24

Missing values in engineered World Bank features:
military_expenditure_pct_gdp_change_1y                  27.54
military_expenditure_pct_gdp_lag1                       26.44
military_expenditure_pct_gdp_rolling_3y_mean            22.27
inflation_consumer_prices_annual_pct_change_1y          16.03
inflation_consumer_prices_annual_pct_lag1               15.77
unemployment_total_pct_change_1y                        15.38
unemployment_total_pct_lag1                             15.35
inflation_consumer_prices_annual_pct_rolling_3y_mean    12.71
unemployment_total_pct_rolling_3y_mean                  12.67
gdp_growth_annual_pct_change_1y                          4.95
gdp_growth_annual_pct_lag1                               4.88
gdp_per_capita_current_usd_change_1y                     4.46
gdp_per_capita_current_usd_lag1       

In [3]:
train_mask = df["year"] <= TRAIN_END_YEAR
test_mask = df["year"] > TRAIN_END_YEAR

print("Train period:", df.loc[train_mask, "year"].min(), "-", df.loc[train_mask, "year"].max())
print("Test period:", df.loc[test_mask, "year"].min(), "-", df.loc[test_mask, "year"].max())

print("\nTrain rows:", train_mask.sum())
print("Test rows:", test_mask.sum())

print("\nTrain target distribution:")
print(df.loc[train_mask, TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

print("\nTest target distribution:")
print(df.loc[test_mask, TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

Train period: 1989 - 2016
Test period: 2017 - 2023

Train rows: 5305
Test rows: 1358

Train target distribution:
target_conflict_next_year
0    0.7086
1    0.2914
Name: proportion, dtype: float64

Test target distribution:
target_conflict_next_year
0    0.6753
1    0.3247
Name: proportion, dtype: float64


In [4]:
def evaluate_predictions(experiment_name, model_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "experiment": experiment_name,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def build_models():
    return {
        "Logistic Regression scaled": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            )),
        ]),
        "Decision Tree": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(
                max_depth=4,
                class_weight="balanced",
                random_state=42,
            )),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )),
        ]),
    }

In [5]:
results = []

y_test = df.loc[test_mask, TARGET_COLUMN]
y_pred_persistence = df.loc[test_mask, "organized_violence_exists"]

results.append(
    evaluate_predictions(
        "reference",
        "Persistence baseline",
        y_test,
        y_pred_persistence,
    )
)

for experiment_name, feature_columns in EXPERIMENTS.items():
    X_train = df.loc[train_mask, feature_columns]
    y_train = df.loc[train_mask, TARGET_COLUMN]

    X_test = df.loc[test_mask, feature_columns]
    y_test = df.loc[test_mask, TARGET_COLUMN]

    models = build_models()

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append(
            evaluate_predictions(
                experiment_name,
                model_name,
                y_test,
                y_pred,
            )
        )

results_df = pd.DataFrame(results)
results_df.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp
0,reference,Persistence baseline,0.9072,0.8571,0.8571,0.8571,854,63,63,378
1,temporal_only,Logistic Regression scaled,0.9161,0.8959,0.8390,0.8665,874,43,71,370
2,temporal_only,Decision Tree,0.8895,0.8198,0.8458,0.8326,835,82,68,373
3,temporal_only,Random Forest,0.8991,0.8319,0.8639,0.8476,840,77,60,381
4,temporal_world_bank_raw,Logistic Regression scaled,0.9153,0.8918,0.8413,0.8658,872,45,70,371
5,temporal_world_bank_raw,Decision Tree,0.9013,0.8218,0.8889,0.8540,832,85,49,392
6,temporal_world_bank_raw,Random Forest,0.9094,0.8472,0.8798,0.8632,847,70,53,388
7,temporal_world_bank_engineered,Logistic Regression scaled,0.9183,0.8929,0.8503,0.8711,872,45,66,375
8,temporal_world_bank_engineered,Decision Tree,0.9006,0.8214,0.8866,0.8528,832,85,50,391
9,temporal_world_bank_engineered,Random Forest,0.9116,0.8512,0.8821,0.8664,849,68,52,389


In [6]:
baseline_f1 = results_df.loc[
    results_df["model"] == "Persistence baseline",
    "f1_score"
].iloc[0]

results_with_diff = results_df.copy()
results_with_diff["f1_difference_vs_persistence"] = (
    results_with_diff["f1_score"] - baseline_f1
)

results_sorted = results_with_diff.sort_values(
    by=["f1_score", "recall", "precision"],
    ascending=False,
)

results_sorted.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp,f1_difference_vs_persistence
7,temporal_world_bank_engineered,Logistic Regression scaled,0.9183,0.8929,0.8503,0.8711,872,45,66,375,0.0139
1,temporal_only,Logistic Regression scaled,0.9161,0.8959,0.8390,0.8665,874,43,71,370,0.0094
9,temporal_world_bank_engineered,Random Forest,0.9116,0.8512,0.8821,0.8664,849,68,52,389,0.0092
4,temporal_world_bank_raw,Logistic Regression scaled,0.9153,0.8918,0.8413,0.8658,872,45,70,371,0.0087
6,temporal_world_bank_raw,Random Forest,0.9094,0.8472,0.8798,0.8632,847,70,53,388,0.0060
0,reference,Persistence baseline,0.9072,0.8571,0.8571,0.8571,854,63,63,378,0.0000
5,temporal_world_bank_raw,Decision Tree,0.9013,0.8218,0.8889,0.8540,832,85,49,392,-0.0031
8,temporal_world_bank_engineered,Decision Tree,0.9006,0.8214,0.8866,0.8528,832,85,50,391,-0.0044
3,temporal_only,Random Forest,0.8991,0.8319,0.8639,0.8476,840,77,60,381,-0.0095
2,temporal_only,Decision Tree,0.8895,0.8198,0.8458,0.8326,835,82,68,373,-0.0246


In [7]:
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_TABLES_DIR / "world_bank_engineered_feature_model_results.csv"

results_with_diff.to_csv(results_path, index=False)

print(f"Saved results to: {results_path}")

Saved results to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\world_bank_engineered_feature_model_results.csv


## Interpretação esperada

A pergunta principal deste notebook é:

> A engenharia de features sobre os indicadores World Bank melhora o desempenho em relação ao uso dos indicadores brutos?

A comparação deve considerar a baseline de persistência e o modelo temporal puro como referências obrigatórias.